# 4 · AI Workflows & System Design for Finance
**You leave with:** a Company Screening Engine that combines live SEC data, deterministic filtering, and validated LLM reasoning.

## The pattern of the day

```
INPUT → RETRIEVE → STRUCTURE → REASON → VALIDATE → HUMAN
         (code)     (code)     (model)    (code)    (you)
```

A **workflow** is a fixed plan written by you; the model fills designated steps. Three design rules carry everything:

1. **Code does math; the model does judgment.** Growth rates and filters live in pandas; the model writes grounded prose *about* them.
2. **Validate at the boundary.** Schema-forced output, plus a numeric audit: any figure in the prose that doesn't trace to your inputs gets flagged.
3. **The human gate is the exit.** Nothing is saved or sent without approval.

## SEC EDGAR in 60 seconds

Every US-listed company's filings, free, no key: just identify yourself (`SEC_EDGAR_USER_AGENT` in `.env`). **There:** 10-K/10-Q/8-K/20-F documents + XBRL fundamentals. **Not there:** prices, estimates. Gotchas we hit building this course (details: `session-04-workflows/edgar-cheatsheet.md`): companies drift between XBRL tags across years; foreign filers report IFRS in local currency; most pharma tags **no operating income** at all.

In [ ]:
import sys, os, json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# If you pulled course updates while this kernel was running, pick them up here
# (a no-op on a fresh kernel; saves a restart otherwise):
import importlib
for _n in [n for n in list(sys.modules) if n.startswith("toolkit")]:
    importlib.reload(sys.modules[_n])
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass
HAS_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
print(f"repo root: {ROOT}")
print(f"API key:   {'configured' if HAS_KEY else 'NOT SET - cells that call Claude will be skipped'}")

## Part A: talk to EDGAR

In [ ]:
from toolkit import edgar

fin = edgar.annual_financials("NVDA", n=3)
print(json.dumps(fin, indent=1))   # the whole record: this is your raw material

In [ ]:
# That's a real 10-K speaking. Let's compute metrics IN CODE (rule 1):
def metrics_row(ticker: str) -> dict | None:
    """3-year fundamentals -> one screening row. Returns None if unusable."""
    try:
        fin = edgar.annual_financials(ticker, n=3)
    except edgar.EdgarError as e:
        print(f"  {ticker}: skipped ({e})")
        return None
    rev = fin["revenue"]
    ni = {v["fy_end"]: v["val"] for v in fin["net_income"]}
    if len(rev) < 3 or fin["unit"] != "USD":
        return None
    r0, r1, r2 = (rev[i]["val"] for i in range(3))
    n_now, n_prev = ni.get(rev[2]["fy_end"]), ni.get(rev[1]["fy_end"])
    return {"ticker": fin["ticker"], "company": fin["company"], "fy_end": rev[2]["fy_end"],
            "revenue_bn": round(r2 / 1e9, 2),
            "growth_1y": round(r2 / r1 - 1, 4),
            "cagr_2y": round((r2 / r0) ** 0.5 - 1, 4),
            "net_margin": round(n_now / r2, 4) if n_now is not None else None,
            "net_margin_prior": round(n_prev / r1, 4) if n_prev is not None else None}

metrics_row("NVDA")

**Why NET margin and not operating margin?** Run `edgar.annual_values(edgar.get_company_facts("LLY"), edgar.OPERATING_INCOME_TAGS)` and observe the result: most pharmaceutical companies present no operating subtotal, so the tag does not exist. Net income is always tagged. *Data coverage is an analytical decision; make it explicitly.*

## Part B: LAB: the Company Screening Engine

Universe: 16 industrials + pharma (`session-04-workflows/data/universe.csv`). First run fetches ~16 filings (a minute or two); everything caches.

### Exercise 1: fetch the universe

In [ ]:
import pandas as pd
universe = pd.read_csv(ROOT / "session-04-workflows" / "data" / "universe.csv")

### START CODE HERE ###
rows = [m for t in universe[None]                     # which column holds the tickers?
        if (m := metrics_row(None)) is not None]      # call the fetcher with what?
df = pd.DataFrame(None)                               # build the table from what?
### END CODE HERE ###

print(f"{len(df)} of {len(universe)} companies fetched")
df.round(3)

In [ ]:
# ✅ self-check: run me
assert len(df) >= 14, "expected at least 14 of 16 to fetch - is your loop skipping None rows?"
assert {"growth_1y", "net_margin"} <= set(df.columns)
assert df["net_margin"].abs().max() < 1, "margins should be decimals, not percents"
print("All checks passed ✅")

### Exercise 2: the deterministic screen
Filter: `growth_1y >= min_growth` AND `net_margin >= min_margin`; if `require_improving`, also `net_margin > net_margin_prior`. Sort by growth, descending.

In [ ]:
def apply_screen(df, min_growth=0.08, min_margin=0.10, require_improving=False):
### START CODE HERE ###
    mask = (df[None] >= min_growth) & (df[None] >= min_margin)
    if require_improving:
        mask &= df["net_margin"] > df[None]               # improving vs which column?
    return df[mask].sort_values(None, ascending=False)    # rank the shortlist by what?
### END CODE HERE ###

shortlist = apply_screen(df)
print("Shortlist:", ", ".join(shortlist["ticker"]))
shortlist.round(3)

In [ ]:
# ✅ self-check: run me
assert len(shortlist) >= 1, "empty shortlist with default criteria - check your mask logic"
assert (shortlist["growth_1y"] >= 0.08).all() and (shortlist["net_margin"] >= 0.10).all()
tight = apply_screen(df, min_growth=0.10, require_improving=True)
assert len(tight) <= len(shortlist), "tighter criteria cannot grow the shortlist"
print("All checks passed ✅  Now play: change the criteria and watch the shortlist move.")

### Exercise 3: grounded rationales + the numeric audit

This exercise calls the API, so it needs your key configured.

The output contract is declared with **Pydantic**, the industry-standard Python validation library (the `anthropic` SDK is itself built on it): a class per object, a typed field per key. `llm.ask_pydantic` forces the model's output through that schema and returns **typed objects**, so downstream code reads `r.ticker` instead of digging into a dict, and a malformed reply fails loudly at the boundary instead of quietly downstream.

ONE call for the whole shortlist; the model may use **only** the metrics you send. Then audit its prose: any number that does not trace back to your table gets flagged (`toolkit/verify.py`).

**Read the flags as a shortlist, not an accusation.** A clean run typically flags a few figures anyway, and they are usually the model's own derived ratios: "roughly 3.5 times", "a gap of 6 percentage points". The checker compares against the numbers you supplied; it cannot perform derivations, so it cannot distinguish a legitimate calculation from an invention. It hands both to you. That is the honest boundary of this kind of automation, and it is still valuable: it reduces what you must verify from an entire page of prose to three or four numbers.

In [ ]:
from pydantic import BaseModel
from toolkit import llm, verify

class Rationale(BaseModel):
    ticker: str
    observation: str

class RationaleSet(BaseModel):
    rationales: list[Rationale]

if HAS_KEY and len(shortlist):
### START CODE HERE ###
    result = llm.ask_pydantic(
        "Write a 2-3 sentence investment observation per company from these screening "
        "metrics (growth/margins as decimals). Use ONLY these metrics - no outside "
        f"knowledge, no new numbers:\n<metrics>\n{shortlist.to_json(orient='records')}\n</metrics>",
        None,                                             # force which output model?
        system="You are screening companies for an investment committee. Grounded, direct, no hype.")
    source_vals = [v for r in shortlist.to_dict("records") for v in r.values()
                   if isinstance(v, (int, float))]
    for r in result.rationales:
        flags = verify.novel_numbers(None, None)          # audit WHICH text against WHICH numbers?
        mark = f"  ⚠️ untraceable: {flags}" if flags else "  ✅ grounded"
        llm.show(f"{r.observation}{mark}", title=r.ticker)
### END CODE HERE ###
    assert isinstance(result, RationaleSet), "typed at the boundary: this is what Pydantic buys you"
    print(f"\nEvery rationale arrived as a typed Rationale object. Token usage: {llm.usage_summary()}")
else:
    print("No API key (or empty shortlist) - the deterministic screen above is still the deliverable core.")

### Exercise 4: train a model to forecast revenue

So far the AI in this course has been **generative**: a language model producing text. The other half of AI is **predictive**: a model whose parameters are *trained* on observed data, and whose quality is *measured* on data it has not seen. This exercise uses the simplest trainable model there is, a least-squares trend line, on six years of real revenue from the filings.

The discipline is the entire lesson, and it transfers unchanged to any model, however sophisticated:

1. **Train** on every year except the last.
2. **Test** on the held-out last year: the error against a value the model never saw is your honest measure of quality.
3. Only then **retrain on everything and forecast** the next year.

A forecast without a measured error is an opinion. This is also the difference from asking the language model to forecast: an LLM's guess cannot be backtested per run; a trained model can.

In [ ]:
import numpy as np

def forecast_revenue(revenues):
    """Least-squares trend: train without the last year, measure on it, then forecast.

    Returns (holdout_error, next_year_forecast)."""
    years = np.arange(len(revenues), dtype=float)
    rev = np.asarray(revenues, dtype=float)
### START CODE HERE ###
    slope, intercept = np.polyfit(years[None], rev[None], 1)   # train WITHOUT the last year
    predicted_last = slope * years[-1] + intercept              # predict the held-out year
    holdout_error = abs(None - rev[-1]) / rev[-1]               # |prediction - truth| / truth
    slope, intercept = np.polyfit(years, rev, 1)                # now retrain on ALL years
    next_forecast = slope * (None + 1) + intercept              # one year beyond the last
### END CODE HERE ###
    return holdout_error, next_forecast

In [ ]:
# ✅ self-check: run me (offline). On perfectly linear data the model must be near-perfect.
err, fc = forecast_revenue([100, 110, 120, 130, 140, 150])
assert err < 0.01, "on linear data the holdout error must be ~0 - did you train WITHOUT the last year?"
assert abs(fc - 160) < 1, "the next value on a +10-per-year trend is 160"
print("All checks passed ✅  A trained, tested, honest forecaster. Now real companies:")

In [ ]:
for t in ["PG", "ETN", "NVDA", "GE"]:
    revs = [v["val"] for v in edgar.annual_financials(t, n=6)["revenue"]]
    err, fc = forecast_revenue(revs)
    print(f"{t:5} last actual {revs[-1]/1e9:7,.1f}bn | holdout error {err:5.0%} | "
          f"FY+1 forecast {fc/1e9:7,.1f}bn {'<- read the error before the forecast' if err > 0.15 else ''}")

*Observation.* The error column is the product, more than the forecast column, and each row teaches a different lesson (your exact numbers will vary as new filings arrive):

- **Procter & Gamble and Eaton** come out with **single-digit errors**: steady businesses in a linear regime, where this model class applies and the forecast is usable.
- **NVIDIA** comes out around **40%**: a straight line cannot describe exponential growth. The model is not broken; it is honestly reporting that it is the wrong model class for this company.
- **General Electric** comes out worst of all, and no model fixes it: GE spun off its healthcare and energy businesses, so the six revenues are not one company's history. **The premise failed before the model ran** — exactly the kind of fact a fitted line cannot know and an analyst must.

Three professional conclusions carry beyond this exercise:

- **Never quote a forecast without its measured error.** The pair is the deliverable; the number alone is an opinion.
- **A large error is information**: it says the growth regime is nonlinear, or the data is not what you assumed. Model selection starts from measured failure.
- **Prediction and generation divide the labor.** The trained model produces the number and its error; the language model, in the exercise above, explains screened companies in grounded prose. Neither should do the other's job.

## Part C: see the full workflow with the human gate

The complete 5-step version (with memo rendering, a sabotage-able validator, and the approval gate) lives in `session-04-workflows/demo/market_intel_workflow.py`. Run it from the terminal; the dry-run requires no key:

```bash
python session-04-workflows/demo/market_intel_workflow.py NVDA --peers AMD INTC --dry-run
```

Worth trying: open `session-04-workflows/data/example_intel_memo.json`, change one figure, rerun: watch the validator catch your sabotage. Revert with `git checkout`.

## Deliverable checklist

- [ ] All ✅ checks green; you ran the screen with at least two different criteria sets
- [ ] You can answer: why did we screen on NET margin? and: which step would you never delegate to the model?
- [ ] (With key) rationales printed with the numeric-audit marks; anything ⚠️ judged by you
- [ ] Committed to your repo

**Next:** `05-agents.ipynb`: the model makes the plan. With a set of controls.